# ASN.1 grammar: parse and check RSA keys
Run `02_openssl_keys.ipynb` first – it creates the key files used here. Run cells from top to bottom. Modify only cells marked `TODO` and the grammar file; the check cells verify your work.

A DER file only contains anonymous INTEGERs. An ASN.1 grammar gives them names – and the names come with mathematical relations that we can check.

In [1]:
import os

for name in ["RSA_2048_priv.der", "RSA_2048_pub.der"]:
    assert os.path.exists(name), f"{name} is missing - run 02_openssl_keys.ipynb first"

### Task 1: Grammar and private key
- Look at the asn1tools example usage: https://asn1tools.readthedocs.io/en/latest/#example-usage
- The grammar of RSA keys is in [RFC 8017](https://www.rfc-editor.org/rfc/rfc8017), Appendix A.1.1 (`RSAPublicKey`) and A.1.2 (`RSAPrivateKey`) – the same definitions as in RFC 3447.
- **TODO:** complete the file `RSA_pub.asn` (open it from the Jupyter file browser), then run the cell. It compiles the grammar and decodes the RSA private key.

In [2]:
import asn1tools

rsa_spec = asn1tools.compile_files("RSA_pub.asn")
priv = rsa_spec.decode("RSAPrivateKey", open("RSA_2048_priv.der", "rb").read())
print(priv)
assert list(priv) == ["version", "modulus", "publicExponent", "privateExponent", "prime1", "prime2",
                      "exponent1", "exponent2", "coefficient"], "complete RSAPrivateKey in RSA_pub.asn"

{'version': 0, 'modulus': 23661354023887509107841302465307366473340397013106805713484935003233793693274709671890378780161700641927768212413527625217325132152486073370346518015298928378788859341914140196238368901167179875080401247791246264619718411935232361726305487950580941077578082989291227253761210140686946490785326888125281767170308094556930428066837533960882173758771498081120067133742092521591546527310111420189394427153692424432255715792786629786337501796209908475105180038769153154635708374899051920042091602300996901603380090751276883342850147758205453645770579437424156183440155847304669658036401671509050163378504866801915718670607, 'publicExponent': 3, 'privateExponent': 157742360159250060718942016435382443155602646754045371423232900021558624621831397812602525201078004279518454749423517501448834214349907155802310120101992855858592395612760934641589126007781199167202674985274975097464789412901549078175369919670539607183853886595274848358408067604579643271902179254168545114466663948

### Task 2: Public key
`RSA_2048_pub.der` is not an `RSAPublicKey`: it is a `SubjectPublicKeyInfo` – an algorithm identifier plus a `BIT STRING` that contains the `RSAPublicKey`
(see Task 2 in `02_openssl_keys` and Task 2 in `03_der_parsing`). Decode only the bytes inside the `BIT STRING`.

In [3]:
pub_der = open("RSA_2048_pub.der", "rb").read()
pub = rsa_spec.decode("RSAPublicKey", pub_der[24:])
print(pub)
assert pub["modulus"] == priv["modulus"] and pub["publicExponent"] == priv["publicExponent"]

{'modulus': 23661354023887509107841302465307366473340397013106805713484935003233793693274709671890378780161700641927768212413527625217325132152486073370346518015298928378788859341914140196238368901167179875080401247791246264619718411935232361726305487950580941077578082989291227253761210140686946490785326888125281767170308094556930428066837533960882173758771498081120067133742092521591546527310111420189394427153692424432255715792786629786337501796209908475105180038769153154635708374899051920042091602300996901603380090751276883342850147758205453645770579437424156183440155847304669658036401671509050163378504866801915718670607, 'publicExponent': 3}


### Task 3: What the fields mean
Each field of `RSAPrivateKey` is defined by a relation. Complete the checks, with $\lambda(n) = \mathrm{lcm}(p-1, q-1)$:
- `modulus` $n = p \cdot q$ (`prime1` = $p$, `prime2` = $q$)
- `privateExponent` $d$: $e \cdot d \equiv 1 \pmod{\lambda(n)}$
- `exponent1` $= d \bmod (p-1)$, `exponent2` $= d \bmod (q-1)$
- `coefficient` $= q^{-1} \bmod p$

Note: textbooks often use $\varphi(n) = (p-1)(q-1)$. Since $\lambda(n)$ divides $\varphi(n)$, a $d$ computed modulo $\varphi(n)$ also satisfies the check with $\lambda(n)$ –
so the $\lambda(n)$ check works for keys from any tool, while some tools compute $d$ modulo $\lambda(n)$ and then the $\varphi(n)$ check fails.

In [4]:
import math

n, e, d = priv["modulus"], priv["publicExponent"], priv["privateExponent"]
p, q = priv["prime1"], priv["prime2"]

assert n == p * q
assert (e * d) % math.lcm(p - 1, q - 1) == 1
assert priv["exponent1"] == d % (p - 1)
assert priv["exponent2"] == d % (q - 1)
assert priv["coefficient"] == pow(q, -1, p)
print("all checks passed")

all checks passed
